# Policy Severity & T5 Fine-Tuning

Goal: fine-tune a small T5 model (Flan-T5-small) to classify GDPR/CCPA policy updates into **severity levels** (e.g., LOW / MEDIUM / HIGH).

Why we do this:
- The pipeline already ingests and cleans policy texts.
- Severity is a key feature for downstream **risk prioritization** and **dashboarding**.
- Instead of hand-coding rules, we train a light text-to-text model that outputs a severity label for each policy.


In [1]:
import os
os.environ["PYTORCH_MPS_DISABLE"] = "1"  # tell PyTorch not to use MPS in this process

import torch

# Force the device to CPU
device = torch.device("cpu")
print("Torch version:", torch.__version__)
print("Using device:", device)


Torch version: 2.9.1
Using device: cpu


In [2]:
import torch

print("Torch version:", torch.__version__)
print("Has MPS backend:", hasattr(torch.backends, "mps"))
if hasattr(torch.backends, "mps"):
    print("MPS available:", torch.backends.mps.is_available())

Torch version: 2.9.1
Has MPS backend: True
MPS available: True


## Imports

In [3]:
import os
import random
from dataclasses import dataclass

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

import torch


print("torch version:", torch.__version__)
print("MPS available:", getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())

torch version: 2.9.1
MPS available: True


In [39]:
MODEL_NAME = "t5-small"  # or whatever you're using

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cpu")
model.to(device)

print("Using device:", device)


Using device: cpu


## Load labeled data

In [42]:
# 1) Choose the input file  
# DATA_PATH = "../data/processed/cleaned_with_severity_zeroshot_20250719T043204Z.csv"
DATA_PATH = "../data/processed/cleaned_with_topic_and_severity.csv"

# 2) Read the full dataframe (keep all original columns)
df = pd.read_csv(DATA_PATH)

print("Original columns:", df.columns.tolist())
print("Number of rows:", len(df))


Original columns: ['source', 'title', 'link', 'date', 'severity', 'topic']
Number of rows: 160


In [44]:
# Standardize column names
# We don't have a summary column, so just define one as empty
df["summary"] = ""

# 4) Build the text that T5 will see
df["text"] = (
    df["source"].fillna("") + " - " +
    df["title"].fillna("")
)

# 5) Clean severity
df = df.rename(columns={"Severity": "severity", "SEVERITY": "severity"})
df["severity"] = df["severity"].astype(str).str.strip().str.upper()

# 6) Create a *model-only* dataframe that T5 will use
df_model = df[["text", "severity"]].dropna().reset_index(drop=True)

print("\nColumns in df_full:", df.columns.tolist())
print("Columns in df_model:", df_model.columns.tolist())
print(df_model.head())
print(df_model["severity"].value_counts())




Columns in df_full: ['source', 'title', 'link', 'date', 'severity', 'topic', 'summary', 'text']
Columns in df_model: ['text', 'severity']
                                                text severity
0  EDPB - The Italian SA imposes fines of 420 000...     HIGH
1  EDPB - Biometrics for attendance recording. Th...   MEDIUM
2  EDPB - Swedish SA: Administrative fine against...   MEDIUM
3  EDPB - Targeted modifications of the GDPR: EDP...   MEDIUM
4  EDPB - Irish Supervisory Authority fines TikTo...     HIGH
severity
MEDIUM      108
HIGH         46
CRITICAL      4
LOW           2
Name: count, dtype: int64


## Train/Validation Split

In [45]:
# For speed during development we can subsample
# N_SAMPLE = None  # or e.g. 500
# if N_SAMPLE is not None:
#     df = df.sample(N_SAMPLE, random_state=42).reset_index(drop=True)

# Stratified split to keep label balance
train_df, eval_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["severity"]
)

print("Train size:", len(train_df), "Eval size:", len(eval_df))
train_df["severity"].value_counts()


Train size: 128 Eval size: 32


severity
MEDIUM      86
HIGH        37
CRITICAL     3
LOW          2
Name: count, dtype: int64

## Wrap in Hugging Face Datasets

In [46]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True))

train_ds, eval_ds

(Dataset({
     features: ['source', 'title', 'link', 'date', 'severity', 'topic', 'summary', 'text'],
     num_rows: 128
 }),
 Dataset({
     features: ['source', 'title', 'link', 'date', 'severity', 'topic', 'summary', 'text'],
     num_rows: 32
 }))

## Load Tokenizer & Model

In [47]:
# Small T5 model – good tradeoff between speed and quality
MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Pad token:", tokenizer.pad_token, " | EOS token:", tokenizer.eos_token)


/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Pad token: <pad>  | EOS token: </s>


## Preprocessing / Tokenization

We treat this as text-to-text:

Input: Classify severity: <'policy text'> <br>
Target: LOW / MEDIUM / HIGH

In [48]:
MAX_INPUT_LENGTH = 256  # truncate long policies
MAX_TARGET_LENGTH = 5   # "HIGH" etc are short

def preprocess_batch(batch):
    # Add an instruction-style prefix (helps Flan models)
    inputs = [f"Classify the regulatory policy severity as LOW, MEDIUM, or HIGH: {t}" 
              for t in batch["text"]]
    targets = batch["severity"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_ds.map(preprocess_batch, batched=True)
eval_tokenized = eval_ds.map(preprocess_batch, batched=True)

train_tokenized[0]


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

{'source': 'EDPB',
 'title': 'This Data Protection Day, meet the EDPB Chair!',
 'link': 'https://edpb.europa.eu/news/news/2024/data-protection-day-meet-edpb-chair_en',
 'date': '2024-01-26',
 'severity': 'HIGH',
 'topic': 'GDPR enforcement',
 'summary': '',
 'text': 'EDPB - This Data Protection Day, meet the EDPB Chair!',
 'input_ids': [4501,
  4921,
  8,
  8253,
  1291,
  20363,
  38,
  3,
  20573,
  6,
  3,
  21357,
  196,
  6122,
  6,
  42,
  27722,
  10,
  3,
  2326,
  13970,
  3,
  18,
  100,
  2747,
  8009,
  1430,
  6,
  942,
  8,
  3,
  2326,
  13970,
  6651,
  55,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [27722, 1]}

## Data Collator

In [49]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
)


## Metrics (accuracy + macro F1)

In [50]:
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_metric

accuracy_metric = load_metric("accuracy")
f1_metric = load_metric("f1")

SEVERITY_LABELS = ["LOW", "MEDIUM", "HIGH"]

def normalize_label(text: str) -> str:
    """
    Clean up raw decoded model outputs and map them to one of:
    LOW, MEDIUM, HIGH.
    """
    if text is None:
        return "MEDIUM"
    t = str(text).upper().strip()

    # Handle weird merges like "HIGHMEDIUM", newline issues, etc.
    for lab in SEVERITY_LABELS:
        if lab in t:
            return lab

    # Fallback if nothing matches
    return "MEDIUM"


def postprocess_text(preds, labels):
    preds = [normalize_label(p) for p in preds]
    labels = [normalize_label(l) for l in labels]
    return preds, labels


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # If preds is a tuple (logits, …), keep only the first element
    if isinstance(preds, tuple):
        preds = preds[0]

    # For seq2seq, Trainer can give:
    # - logits: (batch, seq_len, vocab_size)  -> take argmax
    # - token IDs: (batch, seq_len)
    if preds.ndim == 3:
        preds_ids = preds.argmax(-1)
    else:
        preds_ids = preds

    # Decode predictions and labels into strings
    decoded_preds = tokenizer.batch_decode(
        preds_ids,
        skip_special_tokens=True
    )

    labels_ids = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(
        labels_ids,
        skip_special_tokens=True
    )

    # Normalize to LOW / MEDIUM / HIGH
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # Use sklearn (works with string labels)
    acc = accuracy_score(decoded_labels, decoded_preds)
    f1 = f1_score(decoded_labels, decoded_preds, average="macro")

    return {
        "accuracy": acc,
        "f1_macro": f1,
    }




## Training Arguments

In [51]:
OUTPUT_DIR = "../models/t5_policy_severity"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True
)

/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Trainer

In [52]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer

## Train

In [53]:
train_result = trainer.train()
train_result

/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.207500,0.237397,0.750000,0.525926
2,0.284500,0.213858,0.718750,0.418182
3,0.472100,0.211188,0.750000,0.525926


/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=96, training_loss=0.4115307964384556, metrics={'train_runtime': 18.5422, 'train_samples_per_second': 20.71, 'train_steps_per_second': 5.177, 'total_flos': 7284584153088.0, 'train_loss': 0.4115307964384556, 'epoch': 3.0})

## Evaluate

In [54]:
metrics = trainer.evaluate()
metrics

/Volumes/Personal Drive/GitHub/gdpr-ccpa-risk-pipeline/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.2373967468738556,
 'eval_accuracy': 0.75,
 'eval_f1_macro': 0.5259259259259259,
 'eval_runtime': 0.3108,
 'eval_samples_per_second': 102.961,
 'eval_steps_per_second': 25.74,
 'epoch': 3.0}

## Use the model on all policies

In [55]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 8

def predict_severity(texts, batch_size=8):
    model.eval()
    preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]

        inputs = [f"Classify severity: {t}" for t in batch_texts]

        enc = tokenizer(
            inputs,
            max_length=MAX_INPUT_LENGTH,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        #  Gets the device your model is on
        device = next(model.parameters()).device 


        # Then in the predict_severity function:
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=MAX_TARGET_LENGTH,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded = [d.strip().upper() for d in decoded]
        preds.extend(decoded)

    return preds


In [56]:
all_texts = df["text"].tolist()
pred_severity = predict_severity(all_texts, batch_size=8)
len(pred_severity), pred_severity[:5]


(160, ['EDPB', 'EDPB', 'EDPB - SWEDISH SA', 'EDPB', 'EDPB - IRISH SUPERVISOR'])

## Attach predictions and save CSV

In [57]:
print("Available columns:", df.columns.tolist())

Available columns: ['source', 'title', 'link', 'date', 'severity', 'topic', 'summary', 'text']


In [58]:
df["severity_t5"] = pred_severity

from datetime import datetime
ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

OUT_PATH = f"../data/processed/policies_with_t5_severity_{ts}.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

df.to_csv(OUT_PATH, index=False)

print("Saved policies with T5 severity to:", OUT_PATH)
df[["title", "severity", "severity_t5"]].head()


Saved policies with T5 severity to: ../data/processed/policies_with_t5_severity_20251128T000046Z.csv


,title,severity,severity_t5
0,The Italian SA imposes fines of 420 000 EUR on...,HIGH,EDPB
1,Biometrics for attendance recording. The Itali...,MEDIUM,EDPB
2,Swedish SA: Administrative fine against the Eq...,MEDIUM,EDPB - SWEDISH SA
3,Targeted modifications of the GDPR: EDPB & EDP...,MEDIUM,EDPB
4,Irish Supervisory Authority fines TikTok €530 ...,HIGH,EDPB - IRISH SUPERVISOR


## How this T5 notebook fits the overall GDPR/CCPA Risk Pipeline

1. **Airflow DAG** calls:
   - `fetch_policy_data.py` to scrape / ingest policy feeds.
   - `process_policy_data.py` to clean & normalize text into `cleaned_policies.csv`.

2. A **labeling step** (manual or zero-shot notebook) creates a small labeled file:
   `policy_severity_labeled.csv`.

3. This **T5 fine-tuning notebook**:
   - Reads `policy_severity_labeled.csv`.
   - Fine-tunes `flan-t5-small` to map policy text → severity (LOW/MEDIUM/HIGH).
   - Saves the model under `../models/t5_policy_severity`.

4. In production:
   - A lightweight Python script `severity_inference_t5.py` can load the saved model,
     score new policies daily, and write out `policies_with_t5_severity_*.csv`.

5. The **forecasting notebook** can then:
   - Use `severity_t5` as a feature to model trends over time.
   - Feed aggregated metrics (e.g., count of HIGH-severity events per month) into
     Tableau dashboards for risk monitoring.
